# Harry Potter API paths (`main.py`)

This notebook exercises the **`/harry-potter`** routes from **`app/main.py`** using FastAPI **`TestClient`** (in-process; no separate `uvicorn` process).

**Note:** Cells that **`POST`**, **`PUT`**, or **`DELETE`** modify **`data/harry_potter_data.json`**. Run create/update/delete cells once, or restore the file from git if you repeat them.

In [ ]:
from fastapi.testclient import TestClient

from app.main import app

client = TestClient(app)

# Sample id from bundled JSON (Harry Potter)
SAMPLE_CHARACTER_ID = "103cb127-8bb3-4a34-9fd3-c193a9a6cf54"

## `GET /harry-potter`

List/search with optional query parameters: `first_name`, `last_name`, `house_name` (equality template).

In [ ]:
resp = client.get("/harry-potter")
assert resp.status_code == 200, resp.text
data = resp.json()
assert "items" in data
print("count:", len(data["items"]))
data["items"][:3]

In [ ]:
resp = client.get("/harry-potter", params={"house_name": "Gryffindor"})
assert resp.status_code == 200
items = resp.json()["items"]
assert all(r["house_name"] == "Gryffindor" for r in items)
len(items)

## `GET /harry-potter/{character_id}`

Returns **`404`** if the character does not exist.

In [ ]:
resp = client.get(f"/harry-potter/{SAMPLE_CHARACTER_ID}")
assert resp.status_code == 200
resp.json()

In [ ]:
missing = client.get("/harry-potter/00000000-0000-0000-0000-000000000000")
assert missing.status_code == 404
missing.json()

## `POST /harry-potter`

Creates a character; server assigns **`id`** if omitted. Response body is the new id (JSON string).

In [ ]:
payload = {
    "first_name": "Notebook",
    "last_name": "TestCharacter",
    "house_name": "Ravenclaw",
}
resp = client.post("/harry-potter", json=payload)
assert resp.status_code == 200, resp.text
new_id = resp.json()
assert isinstance(new_id, str)
new_id

## `PUT /harry-potter/{character_id}`

Updates by id; **`400`** if the character does not exist.

In [ ]:
# Uses `new_id` from the POST cell above — run that cell first.
update_body = {
    "first_name": "Notebook",
    "last_name": "TestCharacter",
    "house_name": "Hufflepuff",
}
resp = client.put(f"/harry-potter/{new_id}", json=update_body)
assert resp.status_code == 200
assert resp.json() == {"updated": 1}
client.get(f"/harry-potter/{new_id}").json()

## `DELETE /harry-potter/{character_id}`

Returns **`{"deleted": 0}`** or **`{"deleted": 1}`**.

In [ ]:
resp = client.delete(f"/harry-potter/{new_id}")
assert resp.status_code == 200
assert resp.json()["deleted"] == 1

gone = client.get(f"/harry-potter/{new_id}")
assert gone.status_code == 404
gone.json()